# Data Cleaning and Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# Set visualization style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Load raw data
df = pd.read_csv("../data/stage1_loaded_data.csv")
print(f"Initial Dataset Shape: {df.shape}")
df.head()

## 1. Missing Value Analysis
Identifying and handling null values to maintain data integrity.

In [ ]:
null_counts = df.isnull().sum()
print("Missing Values by Column:")
print(null_counts[null_counts > 0])

# Visualize Missing Values
if null_counts.sum() > 0:
    null_pct = (null_counts / len(df)) * 100
    null_pct[null_pct > 0].plot(kind='bar', color='salmon')
    plt.title("Percentage of Missing Values")
    plt.ylabel("Percentage (%)")
    plt.show()

In [ ]:
# Handle missing values
df['user_name'] = df['user_name'].fillna("Anonymous")
df['review_text'] = df['review_text'].fillna("")

# Drop columns with too many missing values or irrelevant for detection
if 'developer_reply_date' in df.columns:
    df = df.drop(columns=['developer_reply_date'])

print(f"Dataset Shape after Null Handling: {df.shape}")

## 2. Duplicate Detection
Duplicate reviews are a common sign of spam/bot activity in fake review detection.

In [ ]:
duplicates = df.duplicated(subset=['review_text', 'user_name']).sum()
print(f"Found {duplicates} potential duplicate reviews.")

# Remove exact duplicates while keeping the first occurrence
df = df.drop_duplicates(subset=['review_text', 'user_name'], keep='first')
print(f"Dataset Shape after Duplicate Removal: {df.shape}")

## 3. Text Normalization and Cleaning
Preparing text for analysis and stripping unnecessary characters.

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower() # Lowercase
    text = re.sub(r'\s+', ' ', text).strip() # Multi-space to single space
    return text

df['cleaned_review'] = df['review_text'].apply(clean_text)

# Drop reviews that became empty after cleaning
df = df[df['cleaned_review'] != ""]
print(f"Final Clean Dataset Shape: {df.shape}")

## 4. Feature Engineering for Analysis
Adding columns that help us visualize the data distribution.

In [ ]:
df['review_len'] = df['cleaned_review'].apply(len)
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))
df['review_date'] = pd.to_datetime(df['review_date'])

df[['review_len', 'word_count']].describe()

## 5. Visualizations: Confirming Data Health
These plots show the distribution of ratings and content, helping us verify that the data is ready for detection models.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# 1. Rating Distribution
sns.countplot(x='rating', data=df, ax=ax1, hue='rating', palette='viridis', legend=False)
ax1.set_title("Distribution of Ratings")
ax1.set_xlabel("Rating (1-5)")
ax1.set_ylabel("Count")

# 2. Review Length Distribution
sns.histplot(df['review_len'], bins=50, kde=True, ax=ax2, color='teal')
ax2.set_title("Distribution of Review Length (Characters)")
ax2.set_xlabel("Length")
ax2.set_ylabel("Frequency")
ax2.set_xlim(0, df['review_len'].quantile(0.95)) # Zoom in on the bulk of reviews

plt.tight_layout()
plt.show()

In [ ]:
# 3. Top Apps by Review Volume
top_apps = df['app_name'].value_counts().head(10)
sns.barplot(y=top_apps.index, x=top_apps.values, hue=top_apps.index, palette='mako', legend=False)
plt.title("Top 10 Apps by Review Volume")
plt.xlabel("Number of Reviews")
plt.ylabel("App Name")
plt.show()

## 6. Saving Cleaned Data
Exporting the processed dataset for stage 3.

In [ ]:
df.to_csv("../data/stage2_cleaned_data.csv", index=False)
print("Stage 2 Cleaning Complete. Data saved to 'data/stage2_cleaned_data.csv'")